# Exercises on Information Retrieval

<img src="images/gandalf.jpg" width="600" />

## What is Information Retrieval?

- The process of obtaining relevant information from a large repository of data.
- Involves finding material (usually documents) that satisfies an information need.

So, there are already three important concepts:
- documents as unit of information
- queries: the user need
- relevance: a measure of how well a document meets the user need


## Very Brief History of Information Retrieval

- Shannon's Information Theory (1948):
    Claude Shannon's groundbreaking work on information theory laid the foundation for understanding how information can be quantified and transmitted.

- 1960s: Introduced vector space model, term frequency-inverse document frequency (TF-IDF), and relevance feedback mechanisms.

- 1990s: Rise of the World Wide Web
    - Search Engines
    - PageRank Algorithm (1996):
- 2000s: 
    - Integration of machine learning techniques and natural language processing (NLP) for more accurate and context-aware retrieval. 
    - Personalization and Recommendation Systems using user data.

Today we'll do a few simple information retrieval operations for searching over a sample of British Library books

In [7]:
#download dataset using wget store it data folder
#url: https://github.com/Living-with-machines/dhoxss-text2tech/blob/b760f54142bd4833ecc62d0b00805615baf20884/Sessions/data/bl_books_sample.csv

import wget
import os

url = 'https://github.com/Living-with-machines/dhoxss-text2tech/blob/b760f54142bd4833ecc62d0b00805615baf20884/Sessions/data/bl_books_sample.csv'
filename = url.split('/')[-1]

if not os.path.exists('data'):
    os.makedirs('data')

filepath = os.path.join('data', filename)
if os.path.exists(filepath):
    print(f'{filepath} already exists')
else:
    wget.download(url, filepath)
    print(f'Downloaded {filename} to {filepath}')


Downloaded bl_books_sample.csv to data/bl_books_sample.csv


In [1]:
import pandas as pd

# Here we load a sample of 1000 books from a British Library dataset
df = pd.read_csv('data/bl_books_sample.csv')

len(df)

10000

In [2]:
# Let's see the first few rows
df.head()

,Unnamed: 0,record_id,date,raw_date,title,place,empty_pg,text,pg,mean_wc_ocr,...,all_names,Publisher,Country of publication 1,all Countries of publication,Physical description,Language_1,Language_2,Language_3,Language_4,multi_language
0,0,212661,1805-01-01,1805,The History of the Orkney Islands: in which is...,Edinburgh,False,I,3,0.120,...,"Barry, George [person]",NaN,Scotland,Scotland,"viii, 509 pages (4°)",English,NaN,NaN,NaN,False
1,1,212661,1805-01-01,1805,The History of the Orkney Islands: in which is...,Edinburgh,False,"I .,1.1. .1.. .1 by .\CmislAlWSsr?EaMilniTgltl...",10,0.264,...,"Barry, George [person]",NaN,Scotland,Scotland,"viii, 509 pages (4°)",English,NaN,NaN,NaN,False
2,2,212661,1805-01-01,1805,The History of the Orkney Islands: in which is...,Edinburgh,False,THE HISTORY of the ORKNEY ISLANDS: IN which is...,11,0.627,...,"Barry, George [person]",NaN,Scotland,Scotland,"viii, 509 pages (4°)",English,NaN,NaN,NaN,False
3,3,212661,1805-01-01,1805,The History of the Orkney Islands: in which is...,Edinburgh,False,"TO THE RIGHT HONOURABLE LOIB BUIBAS. MY LORD, ...",13,0.786,...,"Barry, George [person]",NaN,Scotland,Scotland,"viii, 509 pages (4°)",English,NaN,NaN,NaN,False
4,4,212661,1805-01-01,1805,The History of the Orkney Islands: in which is...,Edinburgh,False,"SUBSCRIBERS' NAMES. A Eyre, James, Esq. Elder,...",15,0.628,...,"Barry, George [person]",NaN,Scotland,Scotland,"viii, 509 pages (4°)",English,NaN,NaN,NaN,False


### ✏️ Exercise 1

Search a single term in the text of the articles and retrieve the top 5 articles, ranked based on the frequency of that term

In [3]:
term = 'war'
df['count'] = df.text.str.count(term)
df = df.sort_values('count', ascending=False)
print(df[['text', 'count']].head(5))

                                                   text  count
1765  ANTONY AND CLEOPATRA. 229 Did practife on my f...      9
1343  HISTORY OF ENGLAND. 338 c-H A p. James Douglas...      8
5370  SWA SWA 443 Swagger. H.4. S.P. ii.4. three t. ...      8
5222  REV RHE 295 Reverfe (v). T. And. iii.i. L. i.i...      7
1502  EDWARD III. 497 equivalent to near five shilli...      7


In [4]:
# print the content of the top article
print(df.iloc[0].text)

ANTONY AND CLEOPATRA. 229 Did practife on my ftate,8 your being in Egypt Might be my queftion.9 Ant. How intend you, pradtis'd? C^es. Yon may be pleas'd to catch at mine intent, By what did here befal me. Your wife, and bro- ther, Made wars upon me; and their conteftation Was theme for you, you were the word of war." 5 Did praflife on my Jlate, ] To praBiJe means to employ unwar rantable arts or ftratagems. So, in The Tragedie of Anionic, done into Englifh by the countefs of Pembroke, i5g5 : " nothing kills me fo " As that I do my Cleopatra fee " Pratlije with Cxfar." See Vol. VI. p. 187, n. 5. Steevens. 9 queflion. ] i. e. My theme or fubject of convetfation. So again in this fcene : " Out of our queftion wipe him." See Vol. X. p. 107, n. 4. Malone ' ■- their conteftation Was theme for you, you were the word of war.] The only meaning of this can be, that the war, which Antony's wife and brother made upon Caefar, was theme for Antony too lo make war; or was the occalion why he did make

### Issue: Longer documents are ranked higher

How do we address this?


<img src="images/long_doc.jpg" width="300" />

### ✏️ Exercise 2

Normalise word counts by the length of the document

In [5]:
df['norm_count'] = df['count'] / df['text'].str.len()
df = df.sort_values('norm_count', ascending=False)
print(df[['text', 'count','norm_count']].head(5))

                                                   text  count  norm_count
8795  177 All mild, amid the route profane, The holy...      5    0.006435
8652  34 " We'll teach thee the Runic rhyme, teach t...      5    0.005599
2502  131 to be sure, very probable that, if I had s...      5    0.005269
3955  131 to be sure, very probable that, if I had s...      5    0.005269
1765  ANTONY AND CLEOPATRA. 229 Did practife on my f...      9    0.005190


### How to search for multiple keywords?

We have multiple options:
- searching for a phrase
- searching for a series of keywords and combine the counts


### ✏️ Exercise 3
Search for an entire phrase

In [6]:
terms = 'the elephant is the most gentle'
df['count'] = df.text.str.count(terms)
df['norm_count'] = df['count'] / df['text'].str.len()
df = df.sort_values('norm_count', ascending=False)
print(df[['text', 'count','norm_count']].head(5))

                                                   text  count  norm_count
7827  THE HISTORY OF ANIMALS. BOOK IX. 410 CHAPTER X...      1    0.000915
8795  177 All mild, amid the route profane, The holy...      0    0.000000
9856  PREVENTING DOWER. 5 indemnity from the vendor ...      0    0.000000
5833  142 Lost in amaze, I turn'd my steps aside, Wh...      0    0.000000
9990  107 whelmed by them ; but the fervor of con te...      0    0.000000


In [7]:
# print the content of the top article
print(df.iloc[0].text)

THE HISTORY OF ANIMALS. BOOK IX. 410 CHAPTER XLVI. Or all savage animals, hoAvever, the elephant is the most gentle and mild; for he may be instructed in, and possesses a knowledge of many things ; since he is taught to prostrate himself before the king. The senses, also, of the elephant are very acute, and he excels [other quadrupeds] in sagacity. But he does not again meddle Avith the fe male with Avhich he copulates, and which he causes to be pregnant. Some, likewise, say that the elephant lives two hundred, and others that he lives a hundred and t\yenty years, and the females live nearly as many years as the males. It, is, also, asserted that they arrive at their acme about the sixtieth year of their age ; and that they are im patient of the Avinter and cold. The elephant, also, loves rivers, though he is not a river animal, since he dwells near the banks of rivers. He, likeAvise, walks through the water, and proceeds into it as long as his proboscis can keep above it; for he blows

### ✏️ Exercise 4

Search for multiple terms at once weighting them for the length of the documents

In [8]:
terms = ['the','power', 'of','democracy']
df['count'] = df.text.str.count('|'.join(terms))
df['norm_count'] = df['count'] / df['text'].str.len()
df = df.sort_values('norm_count', ascending=False)
print(df[['text', 'count','norm_count']].head(5))

                                                   text  count  norm_count
7764  BOOK IX. THE HISTORY OF ANIMALS. 347 of the el...     83    0.047374
6545  158 Hu Gadam, bringing the Race of the Cymry o...     63    0.046945
6548  161 XV. The three happy astronomers (Seromjddi...     45    0.046826
6542  155 The second were the race of the Lolegrwys*...     71    0.046345
8755  137 The troubles their sorrowful days that bef...      9    0.045918


In [9]:
# print the content of the top article
print(df.iloc[0].text)

BOOK IX. THE HISTORY OF ANIMALS. 347 of the elephant is as follows : Getting on the back of certain tame and courageous elephants, they pursue [those savage elephants they wish to take], and when they have caught them, they order the tame elephants to strike them till they are debilitated ; but then the leader of the elephant, leaping on the back of a savage elephant, manages him with his hook or scythe. And after this the elephant rapidly becomes tame and obedient to his governor. As long, therefore, as the rulers of the wild elephants sit on them, they are all of them mild ; but when they descend from them, some of them return to their former ferocity, and others do not. But in order to tame the more ferocious elephants, they bind their fore-legs Avith chains, and thus render them quiet. The hunting, however, is of elephants that are large, and of those that are young. CHAPTER II. Such, therefore, are the friendships and enmities of these animals, arising from their nutriment, and th

### Issue: Some words are more important than others, but how which ones?

A solution for our problems: TF-IDF

<img src="images/tfidf.png"  />

### ✏️ Exercise 5 (Advanced)
Normalise word counts by using TF-IDF and retrieve the top five articles given a query

In [10]:
## given a term, find the top 5 articles that contain the term based on TFIDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize a TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform the 'text' data
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])

# Convert the term "power" into its TF-IDF representation
query_vector = tfidf_vectorizer.transform(["the power of democracy"])

# Compute similarity scores
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = cosine_similarity(query_vector, tfidf_matrix)

# Get top 5 article indices based on similarity scores
top_article_indices = similarity_scores.argsort()[0][-5:]

# Retrieve top 5 articles
top_articles = df.iloc[top_article_indices]

print(top_articles.iloc[0].text)

41 it had been usurped by a faction avIio had abused it ; that he exercised his poAver just ly and moderately ; that he Avas tlie harbin ger of peace, and the greatest officer exist- ug ; and that if he relaxed the reins of go- ■ernmentj. the French would renew the hor- ors of the Revolution. Balsa is a proprie tor of rich mines, Avas one of the leaders of the Brabant hies, and took some part in the unction between Belgium and France; he says, he is a democrat, and dislikes every goA'ernmei-it but democracy; he adores the names of Vergniaux, Guadet, Condorcet, and the famous twenty-two; he thinks that France avouIcI haA'e been the greatest nation m earth, if Bonaparte had not placed him- eif at the helm; that liberty had at first flourished luxuriantly, but had at length I and turned to the worst corruption ! — Thus men of different notions and principles still give themselves the same denomination; and thus Avhile France combines the con-


## Exercises for Thursdays and Fridays

Issue: TF-IDF does not capture semantic information, only frequency. Can we use word embeddings instead?

### ✏️ Exercise 6
Transform each text in a document embedding and then find the 5 most similar to the query

In [11]:
# convert each text to a document embedding using spacy
import spacy
nlp = spacy.load('en_core_web_sm')

# Create a function to convert text to a document embedding
def get_embedding(text):
    return nlp(text).vector

# Apply the function to the 'text' column
df['embedding'] = df['text'].apply(get_embedding)

# Convert the query to a document embedding
query_embedding = get_embedding("the power of democracy")

# Compute similarity scores
from sklearn.metrics.pairwise import cosine_similarity
df['emb_similarity'] = df['embedding'].apply(lambda x: cosine_similarity([query_embedding], [x])[0][0])

# Get top 5 articles based on similarity scores
top_articles = df.sort_values('emb_similarity', ascending=False).head(5)

print(top_articles[['text', 'emb_similarity']])
print(top_articles.iloc[0].text)

                                                   text  emb_similarity
1045  39 CHAP. XL JOHN. Accession of the king — His ...        0.720750
1011  RICHARD I. 5 Palestine, which were more the re...        0.715529
6242  235 prise. The nation in possession of this pa...        0.708208
4523  732 penters, masons, smiths, are wanting in hu...        0.707183
9968  85 struggle is terminated. What a paradox are ...        0.706790
39 CHAP. XL JOHN. Accession of the king — His marriage — -War zvith France — Murder of Arthur duke of Britany — The king expelled the French provinces — The king's quarrel zvith the court of Rome — Cardinal Lang- ton appointed archbishop of Canterbury — Inter- dict of the kingdom — Excommunication of the king — The king's submission to the pope — Discontents of the barons — Insurrection of the barons — Magna Charta- — Renezoal of the civil wars — Prince Lewis called over — Death — and character of the king. THE noble and free genius of the ancients, which chap m